In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from combra import data
from combra.metrics import wdist_vs_n
from combra.metrics.compare import load_rows

types_dict = {
    'Ultra_Co11': 'small grain',
    'Ultra_Co25': 'medium grain',
    'Ultra_Co8':  'medium-small grain',
    'Ultra_Co6_2': 'large grain',
    'Ultra_Co15': 'medium-small grain',
}

# diffit and san were trained on 3 classes, but the two trainings indexed
# classes in different orders, so each generator needs its own class_map.
# Verified against 4_grid_plot.ipynb's GEN_NAME_FOR_PER_MODE.
CLASS_MAP_PER_KIND = {
    'san': {
        'class_0': 'class_Ultra_Co25',
        'class_1': 'class_Ultra_Co11',
        'class_2': 'class_Ultra_Co6_2',
    },
    'diffit': {
        'class_0': 'class_Ultra_Co11',
        'class_1': 'class_Ultra_Co25',
        'class_2': 'class_Ultra_Co6_2',
    },
}

# Per-resolution sources for the 3x3 convergence grid.
# Each row = one resolution; the 'real' H5 doubles as ethalon (largest-N parquet)
# and as the source of the original-data self-convergence curve.
# Diffit h5s match 4_grid_plot.ipynb's SOURCES so wdist values agree with that
# notebook's table (diffit 256 has 10k/class, diffit 512 has 1k/class).
SOURCES = {
    256: {
        'real':     './data/generated_images_h5/o_bc_left_4x_1536_1024x1024_256x256_rgb_N360.h5',
        'san':      './data/generated_images_h5/gen_san_256x256_N100_000.h5',
        'diffit':   './data/generated_images_h5/00017-diffit-256-gpus2-batch192_N10000.h5',
    },
    512: {
        'real':     './data/generated_images_h5/o_bc_left_4x_1536_1024x1024_512x512_rgb_N360.h5',
        'san':      './data/generated_images_h5/gen_san_512x512_N100_000.h5',
        'diffit':   './data/generated_images_h5/00018-diffit-512-gpus4-batch256_N1000.h5',
    },
    1024: {
        'real':     './data/generated_images_h5/o_bc_left_4x_1536_1024x1024_1024x1024_rgb_N360.h5',
    },
}

# N sweep values per (kind, resolution). Bounded by per-class dataset size of
# each H5 — diffit at 512 only has 1k/class, so the sweep stops there.
# All three kinds start at N=100 so curves share the same x-axis start.
# The largest N in each list also serves as the ethalon for that (resolution, kind).
N_VALUES = {
    'real':     {256:  [100, 150, 200, 250, 300, 360],
                 512:  [100, 150, 200, 250, 300, 360],
                 1024: [100, 150, 200, 250, 300, 360]},
    'san':      {256:  [100, 250, 500, 1000, 2500, 5000, 10000],
                 512:  [100, 250, 500, 1000, 2500, 5000, 10000]},
    'diffit':   {256:  [100, 250, 500, 1000, 2500, 5000, 10000],
                 512:  [100, 250, 500, 1000]},
}

# Columns of the 3x3 grid — the 3 grain classes diffit was trained on.
CLASSES = ['class_Ultra_Co11', 'class_Ultra_Co25', 'class_Ultra_Co6_2']

LABELS = {
    'real':     'original',
    'san':      'san',
    'diffit':   'diffit ',
}

ANGLES_ROOT = Path('./data/angles')

# Angle binning step (degrees). Used by both generation and the W-dist computation.
# generate_angles overwrites the parquet, so changing STEP and re-running the
# generation cell drops any previously-stored step.
STEP = 2.0

def sweep_dir(h5_path):
    return ANGLES_ROOT / (Path(h5_path).stem + '_msl5')

def parquet_for(h5_path, n):
    return sweep_dir(h5_path) / f'angles_n{n}.parquet'

def parquet_has_step(parquet_path, step):
    # True iff the parquet exists AND contains rows tagged with the requested step
    # (each parquet row carries meta.step; wdist_vs_n filters on exact match).
    if not parquet_path.exists():
        return False
    try:
        rows = load_rows(str(parquet_path))
    except Exception:
        return False
    return any(float(r['meta']['step']) == float(step) for r in rows)


# Generation

For every (resolution, source) entry in `SOURCES` extract angle distributions at every N in `N_VALUES[kind]`. Output: `./data/angles/<h5-stem>_msl5/angles_n{N}.parquet`.

The check is **step-aware**: a parquet is considered up-to-date only if it contains rows tagged with the current `STEP`. If you change `STEP` and re-run, every parquet that was generated at a different step will be regenerated (overwritten — `generate_angles` does not append). Files already containing `STEP` are skipped.

The largest N per source doubles as the ethalon used by the convergence plot. For san at 100k images this is the slowest step. Originals reuse existing `prep_cache_*_n360.npy` files in the H5 folder, so N=360 is fast.


In [ ]:
for resolution, group in SOURCES.items():
    print(f'\n=== resolution {resolution}x{resolution} ===')
    for kind, h5_path in group.items():
        out_dir = sweep_dir(h5_path)
        ns = N_VALUES[kind][resolution]
        # Regenerate when the parquet is missing OR exists but has no row at the current STEP.
        missing = [n for n in ns if not parquet_has_step(parquet_for(h5_path, n), STEP)]
        if not missing:
            print(f'  [{kind}] all N present at step={STEP} in {out_dir}')
            continue
        stale = [n for n in missing if parquet_for(h5_path, n).exists()]
        if stale:
            print(f'  [{kind}] {h5_path} -> {out_dir}, regenerating N={stale} (parquet exists but lacks step={STEP})')
        fresh = [n for n in missing if n not in stale]
        if fresh:
            print(f'  [{kind}] {h5_path} -> {out_dir}, generating N={fresh}')
        for N in missing:
            ds = data.PobeditDataset(path=h5_path, max_images_num_per_class=N)
            ds.generate_angles(
                save_path=str(out_dir),
                types_dict=types_dict,
                step=[STEP],
                workers=20,
                angles_tol=3,
                min_segment_len=5.0,
                keep_contours=False,
                chunksize=64,
            )


# Visualization

3×3 W-dist convergence grid. Rows = resolution (256, 512, 1024). Columns = grain class. Each cell overlays the W-dist-vs-N curve for the original (self-convergence baseline against the full-N ethalon at that resolution) plus each generator available at that resolution.

The 1024 row currently has only the original baseline — no san/gen_diff at 1024 in the data folder. Diffit appears in the 256 row only.


In [ ]:
def _max_n(parquet_path):
    rows = load_rows(parquet_path)
    return max((int(r['meta']['n_images']) for r in rows), default=0)

WDIST_SCALE = 1
SAVE_PATH = f'wdist_convergence_step{STEP}.png'

# Human-friendly grain labels for subplot titles.
GRAIN_LABEL = {
    'class_Ultra_Co11':  'small grain',
    'class_Ultra_Co25':  'medium grain',
    'class_Ultra_Co6_2': 'large grain',
}

fig, axes = plt.subplots(len(SOURCES), len(CLASSES),
                         figsize=(6 * len(CLASSES), 5 * len(SOURCES)),
                         sharex=True, sharey=True, squeeze=False)

fig.suptitle(f'W-dist convergence (step={STEP}, ×{WDIST_SCALE})', fontsize=17, y=0.995)

for row, (resolution, group) in enumerate(SOURCES.items()):
    real_dir = sweep_dir(group['real'])
    real_parquets = sorted(real_dir.glob('*.parquet'))
    if not real_parquets:
        print(f'[row {resolution}] no ethalon parquets in {real_dir} — skip row')
        continue
    ethalon_path = max(real_parquets, key=_max_n)
    print(f'row {resolution}: ethalon = {ethalon_path}')

    src_records = {}
    for kind, h5 in group.items():
        d = sweep_dir(h5)
        if not d.exists():
            continue
        cmap = CLASS_MAP_PER_KIND.get(kind)
        recs = wdist_vs_n(d, str(ethalon_path), class_map=cmap, step=STEP)
        # wdist_vs_n globs every parquet in the folder; restrict to the N values
        # declared in N_VALUES so leftover parquets at other Ns don't pollute the plot.
        allowed_ns = set(N_VALUES[kind][resolution])
        recs = [r for r in recs if r['n_images'] in allowed_ns]
        if kind == 'real':
            recs = [r for r in recs if r['w_dist'] > 0]
        src_records[kind] = recs

    for col, cls in enumerate(CLASSES):
        ax = axes[row, col]
        for kind, recs in src_records.items():
            pts = sorted((r['n_images'], r['w_dist']) for r in recs if r['class'] == cls)
            if not pts:
                continue
            Ns, ws = zip(*pts)
            ax.plot(Ns, [w * WDIST_SCALE for w in ws], '-o',
                    label=LABELS.get(kind, kind), markersize=8, linewidth=2)

        ax.set_xscale('log')
        ax.grid(True, which='both', linestyle='-', linewidth=0.6, alpha=0.35)
        ax.set_axisbelow(True)
        ax.tick_params(axis='both', labelsize=13)
        ax.set_title(f'{resolution}×{resolution} — {GRAIN_LABEL.get(cls, cls)}', fontsize=15)
        if row == len(SOURCES) - 1:
            ax.set_xlabel('Number of images, N', fontsize=14)
        if col == 0:
            ax.set_ylabel(f'W-dist × {WDIST_SCALE}', fontsize=14)
        ax.legend(fontsize=12, loc='upper right', framealpha=0.9)

plt.tight_layout(rect=(0, 0, 1, 0.985))
plt.savefig(SAVE_PATH, dpi=150, bbox_inches='tight')
print(f'saved: {SAVE_PATH}')
plt.show()
